# Optimizacioni algoritmi u autonomnoj vožnji
## Profesor: Branislav Ivanov
## Studenti: Natalija Ranđelović i Veljko Petković

## 1. Cilj i ideja projekta

Cilj ovog projekta je da se prikaže primena klasičnih numeričkih algoritama optimizacije u kontekstu autonomne vožnje, sa fokusom na **optimizaciju** parametara upravljačkog sistema vozila. Vozilo se kreće po unapred definisanoj stazi, a njegovo ponašanje zavisi od skupa parametara kontrolera. Promenom tih parametara menja se kvalitet vožnje, stabilnost, brzina i preciznost praćenja putanje.

Zadatak optimizacije je da pronađe takve vrednosti parametara koje **minimizuju** definisanu funkciju cilja i obezbeđuju bezbednu, glatku i efikasnu vožnju.

Na ovaj način projekat povezuje oblasti modelovanja sistema, teorije upravljanja i numeričke optimizacije i pokazuje da se problemi autonomne vožnje mogu uspešno rešavati i bez korišćenja metoda mašinskog učenja.

## 2. Modelovanje i upravljanje vozila

Oblast modelovanja i upravljanja obuhvata definiciju vozila, staze i upravljačkog sistema, kao i njihovu međusobnu interakciju tokom simulacije.

### 2.1 Model vozila

Vozilo je modelovano pomoću kinematičkog bicycle modela, koji opisuje kretanje vozila u ravni bez razmatranja dinamičkih efekata poput klizanja. Model koristi diskretno vreme i ograničenja aktuatora, što omogućava realističnu, ali numerički stabilnu simulaciju.

Implementacija modela vozila nalazi se u fajlu `dynamics.py`, gde su definisani:

- struktura stanja vozila
- ograničenja upravljačkih komandi
- funkcija koja vrši jedan simulacioni korak

### 2.2 Model staze i praćenje putanje

Staza je definisana kao centerline, odnosno niz tačaka koje predstavljaju idealnu putanju vozila. Tokom simulacije se računa bočna greška (*cross-track error*), koja predstavlja udaljenost vozila od centerline putanje.

Generisanje staza i geometrijske operacije nad njima (projekcija na stazu, određivanje lookahead tačke) implementirani su u fajlu `track.py`. Ovaj modul omogućava testiranje vozila na različitim oblicima staza, kao što su S-oblik i kružna staza.

### 2.3 Upravljački sistem

Upravljački sistem je sastavljen iz dva podsistema:

1. Pure Pursuit algoritam za upravljanje pravcem
2. PID regulator za kontrolu brzine vozila

Implementacija upravljačke logike nalazi se u fajlu `controllers.py`, gde su definisani:

- PID regulator
- Pure Pursuit proračun ugla volana
- funkcija koja u svakom koraku generiše upravljačke komande

### 2.4 Simulacioni tok

Kompletna simulacija jednog prolaska vozila kroz stazu (*rollout*) realizovana je u fajlu `utils.py`.

Tokom simulacije se:

- inicijalizuje stanje vozila
- izvršava glavna simulaciona petlja
- beleže metričke vrednosti (greška, vreme, glatkoća)
- računa funkcija cilja

## 3. Optimizacija

Oblast optimizacije bavi se izborom parametara upravljačkog sistema koji minimizuju funkciju cilja i obezbeđuju optimalno ponašanje vozila.

### 3.1 Parametri optimizacije

Parametri koji se optimizuju objedinjeni su u vektor parametara $θ$ i obuhvataju:

- parametre Pure Pursuit algoritma
- PID koeficijente
- referentnu brzinu
- ograničenje ugla volana

Definicija parametara, njihovih dozvoljenih opsega i početnih vrednosti nalazi se u fajlu `config.py`.

### 3.2 Prilagodljivi parametri optimizacije
Algoritmi optimizacije u ovom radu poseduju prilagodljive parametre koji utiču na način pretrage prostora rešenja, brzinu konvergencije i stabilnost rezultata. Ovi parametri ne definišu samo broj iteracija, već kontrolišu ravnotežu između eksploracije i eksploatacije.

Kod _Random Search_ algoritma, parametar `iters` određuje broj nasumičnih uzoraka u prostoru pretrage, dok `seed` omogućava reproduktivnost eksperimenata i analizu varijabilnosti rezultata.

Kod _Coordinate Descent algoritma_, parametar `cycles` definiše broj prolazaka kroz sve koordinate problema, dok `gs_iters` kontroliše preciznost jednodimenzionalne optimizacije duž svake koordinate pomoću metode zlatnog preseka.

Kod _CMA-ES_ algoritma, ključni prilagodljivi parametar je početni globalni korak pretrage `cma_sigma`, koji značajno utiče na opseg istraživanja prostora rešenja i brzinu konvergencije. Parametar `cma_iters` određuje trajanje evolutivnog procesa.

### 3.3 Funkcija cilja

Funkcija cilja kombinuje više kriterijuma performansi:

- prosečnu bočnu grešku
- vreme provedeno van staze
- ukupno vreme vožnje
- glatkoću upravljanja

Ukupna cost funkcija definisana je kao:

$J = 4 * mean_CTE + 40 * T_offroad + 0.6 * T + 0.8 * J_steer$

gde je:

- `mean_CTE` – prosečna bočna greška
- `T_offroad` – ukupno vreme van staze
- `T` – ukupno vreme vožnje (ako vozilo ne stigne do cilja: $T = MAX_TIME + 10$)
- `J_steer` – suma apsolutnih promena ugla volana $J_steer = sum(|delta_k - delta_(k-1)|$)

### 3.4 Algoritmi optimizacije

#### 3.4.1 Random Search algoritam

Random Search algoritam nasumično ispituje prostor parametara unutar dozvoljenih granica i zadržava najbolje pronađeno rešenje. Metod ne zahteva gradijent i koristi se kao referentni pristup.

Implementacija se nalazi u fajlu `random_search.py`.

#### 3.4.2 Coordinate Descent i Golden Section metoda

Coordinate Descent optimizuje parametre jedan po jedan, dok su ostali fiksirani. Za optimizaciju pojedinačnih parametara koristi se Golden Section Search metoda, koja efikasno rešava jednodimenzione probleme minimizacije.

Implementacija se nalazi u fajlu `cd_golden_section.py`.

#### 3.4.3 Nelder–Mead metoda

Nelder–Mead metoda je gradijentno-nezavisni algoritam optimizacije koji koristi geometrijski pristup zasnovan na simpleksu u prostoru parametara. Simpleks predstavlja skup od
$n + 1$ tačaka u $n$-dimenzionalnom prostoru, koje se tokom optimizacije transformišu kroz operacije refleksije, ekspanzije, kontrakcije i skupljanja.

Algoritam iterativno poboljšava poziciju simpleksa tako što pomera najlošiju tačku u pravcu boljih rešenja, bez potrebe za izračunavanjem gradijenata ili Hessian matrica. Ova osobina čini Nelder–Mead metodu pogodnom za probleme sa:
- nelinearnom i nediferencijabilnom funkcijom cilja,
- šumom u evaluaciji,
- relativno malim brojem parametara.

U okviru ovog projekta, Nelder–Mead metoda se koristi za optimizaciju parametara upravljačkog sistema autonomnog vozila, pri čemu se funkcija cilja dobija isključivo putem simulacije vožnje. Implementacija algoritma nalazi se u fajlu `nelder_mead.py`.

#### 3.4.4 CMA-ES algoritam

CMA-ES predstavlja **napredni** evolutivni algoritam optimizacije, posebno dizajniran za rešavanje složenih, nelinearnih i nekonveksnih problema u kontinualnom prostoru. Algoritam funkcioniše tako što u svakoj iteraciji generiše populaciju kandidata iz višedimenzionalne normalne distribucije, čiji se parametri adaptivno ažuriraju.

Ključna karakteristika CMA-ES algoritma je adaptacija kovarijacione matrice, koja omogućava algoritmu da uči korelacije između parametara i automatski prilagođava oblik prostora pretrage. Na taj način CMA-ES efikasno istražuje i eksploatiše prostor rešenja, čak i u prisustvu jakih nelinearnosti i međuzavisnosti parametara.

U kontekstu ovog projekta, CMA-ES se pokazuje kao izuzetno robustan metod za optimizaciju parametara kontrolera autonomnog vozila, jer ne zahteva nikakve pretpostavke o obliku funkcije cilja niti o njenoj diferencijabilnosti. Implementacija CMA-ES algoritma realizovana je u fajlu `cma_es.py`.

### 3.5 Benchmark i poredjenje algoritama

Radi objektivnog poređenja performansi različitih algoritama optimizacije, u projekat je dodat poseban benchmark modul koji omogućava upoređivanje algoritama pod istim uslovima i ograničenjima. Cilj benchmarka je da se ispita efikasnost konvergencije, kvalitet dobijenog rešenja i računarska složenost svakog algoritma.

Benchmark procedura koristi jedinstven evaluacioni budžet, izražen kroz maksimalan broj poziva funkcije cilja. Na ovaj način obezbeđeno je fer poređenje algoritama koji se razlikuju po unutrašnjem mehanizmu rada i broju evaluacija po iteraciji. Za svaki algoritam meri se:

- najbolja dostignuta vrednost funkcije cilja,
- ukupan broj evaluacija,
- ukupno vreme izvršavanja,
- da li je vozilo uspešno stiglo do cilja.

U benchmarku su upoređeni sledeći algoritmi:

- Random Search,
- Coordinate Descent sa Golden Section metodom,
- Nelder–Mead simplex metoda,
- CMA-ES (Covariance Matrix Adaptation Evolution Strategy).

Algoritmi Nelder–Mead i CMA-ES predstavljaju naprednije, gradijentno-nezavisne metode optimizacije. Nelder–Mead koristi geometrijsku transformaciju simpleksa u prostoru parametara i pogodan je za probleme male i srednje dimenzionalnosti. CMA-ES je evolucioni algoritam koji adaptivno uči kovarijacionu matricu distribucije uzorkovanja i poznat je po robusnosti u nelinearnim i nekonveksnim problemima.

Benchmark modul je implementiran u fajlu `run_benchmark.py`, koji automatski:

- mapira ukupni evaluacioni budžet na parametre svakog algoritma,
- prikuplja krive konvergencije (best-so-far),
- generiše numeričke rezultate u JSON i CSV formatima,
- i pravi grafike poređenja konvergencije u funkciji broja evaluacija.

Rezultati benchmarka omogućavaju jasan uvid u kompromis između brzine konvergencije i kvaliteta konačnog rešenja, kao i u razlike između jednostavnih heurističkih i naprednih evolutivnih metoda optimizacije. Na ovaj način projekat prevazilazi pojedinačnu demonstraciju algoritama i pruža sistematsku eksperimentalnu analizu.

### 3.6 Pokretanje optimizacije i eksperimenta

Glavna skripta za pokretanje simulacije i optimizacije je `main.py`. Omogućava izbor režima rada (simulacija, random search, coordinate descent ili replay), kao i izbor staze i parametara eksperimenta.

In [1]:
!python -m src.benchmark.run_benchmark --track s --budget 480

Benchmark settings (approx.):
 problem dim n = 6
 eval_budget = 480
 mapped params: {'random': {'iters': 480}, 'cma': {'iters': 53}, 'nm': {'max_iters': 68}, 'cd': {'cycles': 4, 'gs_iters': 20}}
[random] evals=480 time=25.28s best_J=10.997263200837946
[cd] evals=577 time=21.24s best_J=10.9968530238005
[nm] evals=529 time=19.65s best_J=10.991900422560652
[cma] evals=478 time=18.06s best_J=11.002111375578409
Saved numeric results to results\benchmark_1771597781.json
Saved CSV summary to results\benchmark_1771597781_summary.csv
Saved zoom plot to results\benchmark_1771597781_zoom.png
Saved full plot to results\benchmark_1771597781_full.png

Summary:
| algo   |   final_best_J |   evals | time   | reached   |
|--------|----------------|---------|--------|-----------|
| random |        10.9973 |     480 | 25.28s | True      |
| cd     |        10.9969 |     577 | 21.24s | True      |
| nm     |        10.9919 |     529 | 19.65s | True      |
| cma    |        11.0021 |     478 | 18.06s | Tru

## 4. Analiza rezultata benchmarka optimizacionih algoritama

Benchmark eksperiment je sproveden sa ukupnim evaluacionim budžetom od 480 evaluacija funkcije cilja, pri čemu je dimenzionalnost problema iznosila $n = 6$ (šest parametara upravljačkog sistema). Evaluacioni budžet je približno ravnomerno mapiran na sve algoritme kako bi poređenje bilo fer i uporedivo.

### 4.1 Pregled rezultata

Svi algoritmi su uspešno pronašli rešenje u kojem vozilo stiže do cilja, što potvrđuje da je problem optimizacije stabilan i dobro postavljen. Međutim, razlike između algoritama postaju vidljive pri poređenju kvaliteta konačnog rešenja, brzine konvergencije i računarskog vremena.

- **Random Search** postiže solidan rezultat, ali zahteva najveće vreme izvršavanja. Iako uspeva da pronađe prihvatljivo rešenje, njegova neinformisana priroda dovodi do sporije konvergencije i slabije iskorišćenosti evaluacionog budžeta.
- **Coordinate Descent sa Golden Section** metodom daje nešto bolje rešenje od Random Search-a, ali uz veći broj evaluacija nego što je prvobitno planirano. Ovo ukazuje na stabilno poboljšavanje rešenja, ali i na činjenicu da ovaj metod ima višu evaluacionu cenu po ciklusu.
- **Nelder–Mead** metoda ostvaruje najbolju vrednost funkcije cilja među svim testiranim algoritmima. Uz relativno umeren broj evaluacija i kraće vreme izvršavanja, ovaj algoritam pokazuje dobar balans između lokalnog pretraživanja i efikasnosti, što ga čini veoma pogodnim za probleme male i srednje dimenzionalnosti.
- **CMA-ES** algoritam pokazuje najbrže vreme izvršavanja, ali ne dostiže najbolju vrednost funkcije cilja u okviru datog evaluacionog budžeta. Ovo sugeriše da CMA-ES ima snažan potencijal, ali da zahteva veći budžet evaluacija kako bi u potpunosti iskoristio svoje  adaptivne mehanizme.

Rezultati pokazuju da **ne postoji** univerzalno najbolji algoritam, već da performanse zavise od kompromisa između kvaliteta rešenja i računarskih resursa. Nelder–Mead se pokazao kao najbolji izbor u ovom eksperimentu zbog svoje sposobnosti da brzo pronađe kvalitetno lokalno rešenje. CMA-ES, iako robustan i teorijski snažan, u ovom slučaju nije imao dovoljno evaluacija da nadmaši lokalne metode.

## 5. Zaključak

U okviru ovog projekta prikazana je uspešna primena klasičnih algoritama numeričke optimizacije na problem autonomne vožnje. Korišćenjem jednostavnog, ali realističnog modela vozila i determinističkog upravljačkog sistema, pokazano je da se kvalitet vožnje može značajno unaprediti pravilnim izborom parametara kontrolera.

Rezultati potvrđuju da optimizacija bez mašinskog učenja može dati stabilna, interpretabilna i praktično upotrebljiva rešenja. Transparentnost sistema predstavlja posebnu prednost, jer je uticaj svakog parametra jasan i lako analiziran.

Projekat može poslužiti kao osnova za dalja istraživanja, uključujući proširenje na složenije modele vozila, adaptivne kontrolere ili poređenje sa metodama zasnovanim na učenju.